In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

In [15]:
import json

In [2]:
load_dotenv()

True

In [3]:
llm_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
model="llama-3.3-70b-versatile"

In [5]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()



In [13]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query"
                }
            },
            "required": ["query"]
        }
    }
}

In [18]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

## A function-call helper

In [11]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

# 2. Processing one response

In [19]:
import json

def make_call(call):

    function_name = call.function.name
    args = json.loads(call.function.arguments)

    if function_name == "search":
        result = search(**args)
    else:
        raise ValueError(f"Unknown tool: {function_name}")

    return {
        "role": "tool",
        "tool_call_id": call.id,
        "content": json.dumps(result)
    }

In [20]:
question = "I just discovered the course. Can I join it?"

In [21]:
messages = [
    {
        "role": "system",
        "content": instructions
    },
    {
        "role": "user",
        "content": question
    }
]

In [22]:
response = llm_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=[search_tool],
    tool_choice="auto"
)